<a href="https://colab.research.google.com/github/Tanmay-Godse/Financial-Analytics-Data-Pipeline-with-Machine-Learning/blob/main/Run_LLM_on_T4_or_higher_GPU_via_VLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run opensource LLM's with VLLM server on COLAB GPU

## Things to keep in mind before starting:



*   While VLLM Server is running, do not goto **STEP-11** and rerun it, it'll crash the server and will become too technical to restart, instead kill the session, restart the kernal and run **STEP-4.1** first, then follow all the steps in sequence!

*   Not every model from HuggingFace would work here, the base T4_GPU **(Available in free version of colab)** only have ~15GB of VRAM, so plan accordingly.

*   If you have **Colab PRO**, you can leverage higher memory GPU like L4 (approx. 24GB VRAM), A100 (Var_1: approx. 40GB VRAM, Var_2: approx. 80GB VRAM), H100(approx. 80GB VRAM, **Strongest GPU** on colab currently), G4 (RTX A6000 Pro, approx. 96GB VRAM).

##**STEP-1**  Check GPU Stats:

In [101]:
!nvidia-smi

Sat Apr 25 17:38:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             66W /   70W |   12487MiB /  15360MiB |     99%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **STEP-1.1** Solves broken path problems

In [49]:
%cd /content

/content


##**STEP-2** Install UV (Python Package Installer + Manager)

In [29]:
!pip install uv

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.


## **STEP-3** Install Important Libs:

In [30]:
!uv pip install torch torchvision torchaudio openai transformers huggingface_hub accelerate safetensors --index-url https://download.pytorch.org/whl/cu121

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Current directory does not exist


##**STEP-4** Install VLLM (LLM SERVER):

In [50]:
!uv pip install vllm

Using Python 3.12.13 environment at: /usr
Checked 1 package in 161ms


##**(STEP-4.1)** Delete folder only if path is not found in **STEP-8**!

In [51]:
!rm -rf /root/.cache/

##**STEP-5** Download Model from HuggingFace:

In [52]:
!hf download Qwen/Qwen3-4B-Instruct-2507

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 13 files: 100% 13/13 [03:27<00:00, 15.94s/it]
Download complete: : 8.06GB [03:27, 68.4MB/s]              ✓ Downloaded
  path: /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554
Download complete: : 8.06GB [03:27, 38.9MB/s]


##**STEP-6** Check folder size (OPTIONAL)

In [53]:
!du -sh /root/.cache/

7.6G	/root/.cache/


##**STEP-7** Check pressent working directory :

In [54]:
!pwd

/


## **STEP-8** This is the model download path:

In [55]:
%cd ../root/.cache/huggingface/hub

/root/.cache/huggingface/hub


##**(OPTIONAL STEP-1)** Check Current working directory, updated!

In [56]:
!pwd

/root/.cache/huggingface/hub


##**(OPTIONAL STEP-2)** Check all available/downloaded Models from huggingface (to download model, run : 'hf download "model card name" ').

In [57]:
!ls ~/.cache/huggingface/hub/

CACHEDIR.TAG  models--Qwen--Qwen3-4B-Instruct-2507


##**STEP-9** Get Model Snapshot ID (Unique Hash ID provided by HuggingFace)

In [58]:
!ls ~/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/

cdbee75f17c01a7cc42f958dc650907174af0554


##**STEP-10** Set the model path to that unique snapshot (Caution: This snapshot hash ID might get changed, please update model path manually from above cell output [ID], This value will be different for every LLM model!)

In [60]:
model_path = "/root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554"

##**STEP-11** Run VLLM Server in backend and use it via API endpoint

In [102]:
!python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen3-4B-Instruct-2507 \
    --dtype float16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.80 \
    --trust-remote-code \
    --enforce-eager \
    > vllm.log 2>&1 &

## **STEP-11.1** Post check on server stats

In [103]:
!ps aux | grep vllm

root       14569  3.0  9.9 8263804 1323772 ?     Sl   17:29   0:17 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --dtype float16 --max-model-len 1024 --gpu-memory-utilization 0.80 --trust-remote-code --enforce-eager
root       17391 88.0  3.8 4965496 508796 ?      Rl   17:39   0:01 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --dtype float16 --max-model-len 8192 --gpu-memory-utilization 0.80 --trust-remote-code --enforce-eager
root       17402  0.0  0.0   7376  3468 ?        S    17:39   0:00 /bin/bash -c ps aux | grep vllm
root       17404  0.0  0.0   6484  2404 ?        S    17:39   0:00 grep vllm


## **STEP-12** Ping to test Backend Server!
###Wait 3~5 mins after running **STEP-11**
### Till then You can run and check **(OPTIONAL STEP-3)**
### Also you can check GPU RAM consumption in Resources if GUI info required!


In [104]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3-4B-Instruct-2507","object":"model","created":1777138747,"owned_by":"vllm","root":"Qwen/Qwen3-4B-Instruct-2507","parent":null,"max_model_len":1024,"permission":[{"id":"modelperm-bb56ef0efec85cda","object":"model_permission","created":1777138747,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

##**(OPTIONAL STEP-3)** Post checks on running VLLM server

In [107]:
!tail -n 50 vllm.log

(APIServer pid=17391)     return self._loop.run_until_complete(task)
(APIServer pid=17391)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=17391)   File "uvloop/loop.pyx", line 1518, in uvloop.loop.Loop.run_until_complete
(APIServer pid=17391)   File "/usr/local/lib/python3.12/dist-packages/uvloop/__init__.py", line 48, in wrapper
(APIServer pid=17391)     return await main
(APIServer pid=17391)            ^^^^^^^^^^
(APIServer pid=17391)   File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py", line 672, in run_server
(APIServer pid=17391)     await run_server_worker(listen_address, sock, args, **uvicorn_kwargs)
(APIServer pid=17391)   File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py", line 686, in run_server_worker
(APIServer pid=17391)     async with build_async_engine_client(
(APIServer pid=17391)                ^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=17391)   File "/usr/lib/python3.12/contextlib.py", 

## **STEP-13** Command to use the LLM via openAI endpoint created by running VLLM in backend, you can change your desired prompt by editing here : "Explain Schrödinger equation in depth"

###**SUGESTION**
###Until **STEP-12** do not responds correctly, do not hit VLLM API endpoint!

###if crashed, goto **[STEP-4.1]** and redo steps.

In [100]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

resp = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[{"role": "user", "content": "Explain Schrödinger equation in depth"}],
)

print(resp.choices[0].message.content)

The **Schrödinger equation** is the cornerstone of quantum mechanics, providing a mathematical description of how the quantum state of a physical system evolves over time. It replaces classical mechanics in describing the behavior of particles at atomic and subatomic scales. Below is a **deep and comprehensive explanation** of the Schrödinger equation, covering its historical context, mathematical form, physical interpretation, types, solutions, and implications.

---

## 🔍 1. Historical Background

Before the development of quantum mechanics, classical physics (Newtonian mechanics and electromagnetism) failed to explain phenomena such as:

- The discrete energy levels of atoms.
- The photoelectric effect.
- Atomic stability (why electrons don’t spiral into the nucleus).

In 1925–1926, **Erwin Schrödinger** introduced a wave-based formulation of quantum mechanics, which became known as **wave mechanics**. His equation provided a way to describe particles not as point masses, but as **w

#**That's all for now Folks, see you later!**


